In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

model_name  = 'klue/bert-base'
# 한국어 토크나이져
tokenizer = BertTokenizer.from_pretrained(model_name)
# 분류기
model = BertForSequenceClassification.from_pretrained(model_name, num_labels = 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admini

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [2]:
from torch.optim import AdamW
from tqdm import tqdm
# 소규모 훈련 데이터 구축 및 학습(Fine-tuning)
train_texts = [
    "이 영화 진짜 개꿀잼이네요 ㅋㅋㅋ 완전 강추!",   # 긍정
    "배우들 연기가 훌륭하고 스토리가 감동적입니다.", # 긍정
    "돈 아깝네요 최악입니다. 절대 보지 마세요.",     # 부정
    "스토리가 너무 지루하고 결말이 어이없음.",       # 부정
]
train_labels = [1, 1, 0, 0] # 1: 긍정, 0: 부정
# 토크나이징
inputs = tokenizer(train_texts,padding=True, truncation=True,return_tensors='pt').to(device)
labels = torch.tensor(train_labels).to(device)
optimizer =AdamW(model.parameters(), lr = 5e-5)
model.train()
epochs=3

# 허깅페이스에 올라간 모델들은 loss함수를 내장하고 있어서 훈류인지 회귀인지 알아서 알맞은 손실함수를 자동 적용
# 입력과 정답을 주면 알아서....

for epoch in tqdm(range(epochs)):
    optimizer.zero_grad()
    outputs = model(**inputs,labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f'epoch: {epoch+1} /{epochs}  loss : {loss.item()}')

 33%|███▎      | 1/3 [00:00<00:01,  1.20it/s]

epoch: 1 /3  loss : 0.7090270519256592


 67%|██████▋   | 2/3 [00:01<00:00,  1.64it/s]

epoch: 2 /3  loss : 0.40754321217536926


100%|██████████| 3/3 [00:01<00:00,  1.77it/s]

epoch: 3 /3  loss : 0.20814161002635956


In [3]:
# 추론
model.eval()
test_texts = [
    "시간 가는 줄 모르고 봤습니다. 인생작 등극!",
    "도대체 무슨 내용을 말하고 싶은 건지 모르겠네요. 돈버림."
]

test_inputs = tokenizer(test_texts,padding=True, truncation=True, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model(**test_inputs)
    logits = outputs.logits   
    predictions = torch.argmax(logits,dim=-1)
    print(predictions)

tensor([1, 0])
